<a href="https://colab.research.google.com/github/yubou0/LTC_policy3.0_RAG/blob/main/LTC_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#長照 3.0 政策諮詢 RAG 系統

## 1.基本資訊
- **專案動機**:「長照十年計畫 3.0核定本」長達 181 頁，內容包含大量法律術語與行政細節，對一般民眾或照護者而言閱讀門檻極高。本專案透過 RAG (Retrieval-Augmented Generation) 技術，建立一個具備事實根據的問答系統，讓使用者能透過自然語言快速檢索核心政策。
- **Data Sources**:衛生福利部官方發布之《長照十年計畫 3.0 (115-124 年) - 核定本》PDF 文件


## 2.requirements
Python 3.12.12

langchain

langchain-community

langchain-classic

langchain-google-genai

langchain-huggingface

pypdf

chromadb

sentence-transformers

## 安裝RAG所需的關鍵套件

In [ ]:
!pip install -q -U langchain langchain-community langchain-classic langchain-google-genai langchain-huggingface pypdf chromadb sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.3/359.3 kB 37.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ragas 0.1.21 requires langchain<0.3, but you have langchain 1.2.12 which is incompatible.
ragas 0.1.21 requires langchain-community<0.3, but you have langchain-community 0.4.1 which is incompatible.
ragas 0.1.21 requires langchain-core<0.3, but you have langchain-core 1.2.19 which is incompatible.
langchain-openai 0.1.25 requires langchain-core<0.3.0,>=0.2.40, but you have langchain-core 1.2.19 which is incompatible.


## 讀取與切割文件
使用 PyPDFLoader 讀取 181 頁的政策文件，並使用 RecursiveCharacterTextSplitter 將文本切分為 800 字的小片段，並設置 100 字的重疊區間以確保語意不中斷。

In [ ]:
import sys
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 抓PDF (路徑在 /content/)
file_path = "/content/長照十年計畫3.0(115-124年)-核定本.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

# chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
split_docs = text_splitter.split_documents(pages)

print(f"success！total pages：{len(pages)}")
print(f"success！chunks：{len(split_docs)}")

success！total pages：182
success！chunks：246


## 向量化&建立vectorDB
採用 paraphrase-multilingual-MiniLM-L12-v2 嵌入模型。這是一個針對多國語言優化的輕量化模型，並選用 Colab 的 T4 GPU 進行。

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 自動偵測是否可以使用 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用設備: {device}")

# 初始化Embedding模型
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': device}
)

# 建立VectorDB(存在 Colab 暫存目錄)
vector_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("success set vectorDB")

正在使用設備: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


success set vectorDB


## RAG Chain Construction
採用 LangChain 最新版之 create_retrieval_chain 架構，連結 Gemini 1.5 Flash API。透過 System Prompt 的設計，嚴格限制 AI 只能根據檢索到的「文件片段」回答，以避免幻覺

參考資料:(https://github.com/langchain-ai/langchain/issues/33822)

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain v1.0
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. API Key
os.environ["GOOGLE_API_KEY"] = "your key"

# 2. Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 3. Prompt
system_prompt = (
    "你是一位專業的長照政策顧問。你只會回答所擁有的文件，若是文件沒提到，則回覆文件未提及。請根據以下內容回答問題。"
    "內容：{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 使用從 langchain_classic 導入的函數
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(vector_db.as_retriever(), combine_docs_chain)

## 實際QA

In [ ]:
# 5. 測試
query = "醫照整合的具體做法有哪些？"
response = rag_chain.invoke({"input": query}) #提問轉成向量，去DB找接近的

print(f"答：\n{response['answer']}")

答：
根據您提供的文件，醫照整合的具體做法包括：

1.  **推動在宅責任醫師制度：**
    *   由家庭醫師提供以人為中心的整合性健康管理模式，服務內容涵蓋預防保健、健康促進、疾病管理、在宅醫療等。
    *   結合醫院與社區醫療團隊（如診所、居護所、藥局、物理治療所等），及早發現潛在長照需求者。
    *   強化在宅/照護機構的急症照護及安寧療護。
    *   串連社區生活照顧協同單位，協助連結所需的醫療照護服務及長照服務資源，建構全人、全程、全社區的照護服務。

2.  **建構住宿型照護機構住民整合照護模式：**
    *   由醫療團隊進駐住宿型照護機構，採跨專業合作機制共同照護機構住民。
    *   整合醫療、照顧、復能、心理輔導等服務。
    *   以論人計酬的支付概念，促使照護機構主動導入預防保健、減少機構內感染。
    *   建立每位住民個別化的照護措施，提升住民在機構內的健康生活與品質。

3.  **建置智慧化在宅醫療生態系：**
    *   利用數位科技整合醫療照護。

4.  **加速長照服務銜接時效：**
    *   衛福部規劃由醫院成立評估小組，於個案出院日前完成長照需要等級評估。
    *   在照管系統擬定簡易個案照顧計畫後，聯絡長照個案居住地的各縣市照管中心，確保個案出院後能獲得所需長照服務資源。


# mertice
可使用RAGAS去做評估
Ragas 採用 LLM-as-a-Judge 的技術，利用高階模型具備的推理能力，對實作模型進行深度稽核，能精準偵測出人工難以發現的語意偏差。

關鍵指標

- Faithfulness,驗證回答是否嚴格遵守 PDF 原文，確保政策不被誤傳。

- Answer Relevancy,確保系統能精確回答提問，不產生冗餘資訊。

- Context Precision,驗證 Embedding 模型是否能定位核心條文。

## Findings
- 1.LangChain v1.0+ 進行了大規模模組拆分，參考了[此資料](https://github.com/langchain-ai/langchain/issues/33822)才解決
- 2.在 Colab 使用 GPU 進行 Embedding 的速度比本地 CPU 快很多
- 3.RAGAS的概念像是兩個LLM在對抗